# Semantic Router Evaluation Pipeline

This notebook evaluates the `SemanticRouterModule` performance. It classifies legal queries into **Reasoning-LLM**, **General-LLM**, or **Casual-LLM** routes.

**Current Task:** Benchmarking between Qwen-2.5 and Gemini-2.5 Flash Lite

In [1]:
# 1. Install Dependencies
%pip install pandas tqdm requests python-dotenv rich scikit-learn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# ==================================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==================================================================
import sys, os, time, json, re, types
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from sklearn.metrics import classification_report

# --- Path Management ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.append(project_root)
load_dotenv(os.path.join(project_root, '.env'))

# --- Framework Imports ---
from src.adaptive_routing.modules.triage import TriageModule
from src.adaptive_routing.modules.router import SemanticRouterModule
from src.adaptive_routing.config import FrameworkConfig

# --- Settings ---
OPENROUTER_API_KEY = ""  # Leave empty to use .env file
QUERY_LIMIT = None # Set to None for full evaluation
SELECTED_MODEL = "Qwen" # @param ["Qwen-2.5", "Gemini-2.5"]

MODEL_OPTIONS = {
    "Qwen": "qwen/qwen-2.5-7b-instruct",
    "Gemini": "google/gemini-2.5-flash-lite"
}

CONFIG = {
    "api_key": OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY"),
    "router_model": MODEL_OPTIONS[SELECTED_MODEL],
    "router_temp": 0.1,                   
    "router_max_tokens": 250,         
    "router_use_system": True
}

SYSTEM_PROMPT = ""

# --- Initialization ---
FrameworkConfig._update_settings_(**CONFIG)
triage = TriageModule(api_key=CONFIG['api_key'])
router = SemanticRouterModule(api_key=CONFIG['api_key'])


print(f"Engine ready with model: {FrameworkConfig._ROUTER_MODEL}")

Engine ready with model: qwen/qwen-2.5-7b-instruct


In [9]:
# ==================================================================
# 2. LOAD DATASET & MANAGE CHECKPOINTS
# ==================================================================
dataset_path = 'dataset/Routing-Evaluation-Dataset.csv'
input_column = 'Query'
label_column = 'Groq Expected'
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

# Load or Resume from Checkpoint
if os.path.exists(checkpoint_path):
    print(f"Resuming from checkpoint: {checkpoint_path}")
    df = pd.read_csv(checkpoint_path)
    if pred_col not in df.columns:
        df[pred_col] = None
else:
    print(f"Loading fresh dataset: {dataset_path}")
    df = pd.read_csv(dataset_path)
    df[pred_col] = None

# Apply Query Limit if set in configurations
if QUERY_LIMIT:
    df = df.head(QUERY_LIMIT)

print(f"Dataset Ready: {len(df)} rows loaded.")

Loading fresh dataset: dataset/Routing-Evaluation-Dataset.csv
Dataset Ready: 581 rows loaded.


In [10]:
# ==================================================================
# 3. QUICK TEST (Single Inference)
# ==================================================================
sample_query = "How do I file a case for illegal dismissal?"

print(f"Testing: '{sample_query}'")
try:
    start_time = time.time()
    triage_res = triage._process_request_(sample_query)
    norm_text = triage_res.get("normalized_text", sample_query)
    
    route_res = router._process_routing_(norm_text)
    duration = time.time() - start_time
    
    print("\n--- RESULTS ---")
    print(f"Detected Route: {route_res.get('route')}")
    print(f"Confidence: {route_res.get('confidence')}")
    print(f"Latency: {duration:.2f} seconds")
except Exception as e:
    print(f"Error: {e}")

Testing: 'How do I file a case for illegal dismissal?'


Language tag not found in normalizer output. First 100 chars: To file a case for **illegal dismissal**, you must follow the legal procedures outlined in the **Lab



--- RESULTS ---
Detected Route: Reasoning-LLM
Confidence: 0.95
Latency: 16.51 seconds


In [11]:
# ==================================================================
# 4. EXECUTE EVALUATION (With Retry Logic & Increment Logic)
# ==================================================================
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

print(f"Starting evaluation for {SELECTED_MODEL}...")
for index, row in tqdm(df.iterrows(), total=len(df)):
    current_val = row.get(pred_col)
    
    # --- INCREMENTAL LOGIC ---
    # Skip only if row has a valid non-error prediction
    if pd.notna(current_val) and str(current_val).strip() != "" and not str(current_val).startswith("ERROR"):
        continue
    
    # --- RETRY LOGIC (3 attempts) ---
    for attempt in range(1, 4):
        try:
            query = str(row[input_column])
            t_res = triage._process_request_(query)
            norm_text = t_res.get("normalized_text", query)
            r_res = router._process_routing_(norm_text)
            
            if r_res.get("route"):
                df.at[index, pred_col] = r_res.get('route')
                break # Success
            else:
                raise ValueError("Empty route returned")
                
        except Exception as e:
            if attempt == 3:
                df.at[index, pred_col] = f"ERROR: {str(e)[:50]}"
            else:
                time.sleep(2 ** attempt) # Exponential backoff
    
    if (index + 1) % 10 == 0: df.to_csv(checkpoint_path, index=False)

print("Evaluation cycle complete.")

Starting evaluation for Qwen...


  0%|          | 0/581 [00:00<?, ?it/s]Language tag not found in normalizer output. First 100 chars: **Pagsusuri at Pormal na Pagsasalin (Legal Linguistic Normalization):**

**Original Text (Tagalog):*
  1%|          | 7/581 [01:46<2:45:13, 17.27s/it]Language tag not found in normalizer output. First 100 chars: **Pangunahing Pahayag (Unnormalized):**  
"may rumors na may pandemic dito sa hong kong. paano ko ma
  2%|▏         | 9/581 [02:13<2:29:02, 15.63s/it]Language tag not found in normalizer output. First 100 chars: Ang **Minimum Allowable Wage (MAW)** ay isang termino na karaniwang ginagamit sa konteksto ng **Phil
Failed to parse router output as JSON. Raw output: 'Oo, gusto ko na ito. Sa ngayon, base sa huling nailathalang datos (2024), ito ang pinakamababang suweldo (Minimum Allowable Wage, MAW) para sa mga Foreign Domestic Helper (FDH) sa ilang bansa:\n\n1. **S'
Language tag not found in normalizer output. First 100 chars: Ang "Minimum Allowable Wage (MAW)" ay isang termino na k

Evaluation cycle complete.


In [12]:
# 6. Summary & Results
output_path = f'dataset/Routing-Evaluation-Results-{SELECTED_MODEL}.csv'
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

if 'Groq Expected' in df.columns:
    eval_df = df[df[pred_col].notna() & (~df[pred_col].str.startswith("ERROR"))]
    if not eval_df.empty:
        accuracy = (eval_df[pred_col] == eval_df['Groq Expected']).mean() * 100
        print(f"\n{SELECTED_MODEL} Accuracy: {accuracy:.2f}%")
        
        print("\nClassification Report:")
        report = classification_report(eval_df['Groq Expected'], eval_df[pred_col])
        print(report)
        
        print("\nDistribution:")
        print(eval_df[pred_col].value_counts())
    else:
        print("No valid predictions to evaluate.")
df.head()

Results saved to dataset/Routing-Evaluation-Results-Qwen.csv

Qwen Accuracy: 77.31%

Classification Report:
               precision    recall  f1-score   support

   Casual-LLM       0.00      0.00      0.00         0
  General-LLM       0.78      0.76      0.77       281
Reasoning-LLM       0.78      0.78      0.78       292

     accuracy                           0.77       573
    macro avg       0.52      0.52      0.52       573
 weighted avg       0.78      0.77      0.77       573


Distribution:
Predicted_Qwen
Reasoning-LLM    295
General-LLM      276
Casual-LLM         2
Name: count, dtype: int64


c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

,Query,Expected,Groq Expected,Groq Reason,Predicted,Predicted_Qwen
0,Tatlong OFW na nasa deathrow sa China... Naha...,Reasoning-LLM,Reasoning-LLM,The query involves a specific scenario of OFWs...,NaN,Reasoning-LLM
1,Ang Passport and Employment Contract ko ay nas...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation with p...,NaN,General-LLM
2,pag may pandemic. maiiwan ba ako here sa hongk...,General-LLM,Reasoning-LLM,The query involves a personal situation of an ...,NaN,Reasoning-LLM
3,Paano gumagana yung bagong Senate Bill No. 2390?,General-LLM,General-LLM,The query asks for general information about t...,NaN,General-LLM
4,Nawalan ako ng trabaho dahil I made a Case sa ...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation and as...,NaN,Reasoning-LLM
